## 최종 컬럼 명세

시간은 UTC 기준.

| 컬럼 | 컬럼정의 | 생성방법 | 용도 |
| --- | --- | --- | --- |
| `device` | 기기 식별자 | 원본 `device`를 문자열로 사용 | 플레이어 구분·결과 대조 |
| `first_seen` | 최초 관측된 플레이 종료 시각 | 기기별 `played_at` 최솟값 | 관측 구간 기준·시간순 분할 |
| `after5` | 예측 시점 | `first_seen + 5일` | 피처 구간과 결과 구간의 경계 |
| `after15` | 결과 관측 종료 시점 | `first_seen + 15일` | 라벨 구간 경계·관측 가능 대상 선정 |
| `churn` | 이후 10일간 미복귀 여부 | `[after5, after15)`에 기록이 없으면 1, 있으면 0 | 예측 타깃 |
| `play_count` | 첫 5일 플레이 횟수 | `[first_seen, after5)`의 기기별 로그 수 | 모델 입력 피처 |
| `mean_score` | 첫 5일 평균 점수 | 같은 구간의 기기별 `score` 평균 | 모델 입력 피처 |
| `active_duration_days` | 첫 5일 내 활동 간격(일) | 같은 구간의 마지막·첫 플레이 시각 차이를 초로 환산해 86,400으로 나눔 | 모델 입력 피처 |


### 원본 로딩·정제


In [1]:
import pandas as pd

logs = pd.read_csv('../data/raw/rawdata_game1.csv', dtype={'device': 'string'})
# ID 결측과 완전 중복 제거
logs = logs.dropna(subset=['device']).drop_duplicates().copy()
# Unix 초를 UTC 시각으로 변환
logs['played_at'] = pd.to_datetime(logs['time'], unit='s', utc=True)


### 관측 구간·대상 선정


In [2]:
players = (
    logs.groupby('device')['played_at']
    .min()
    .rename('first_seen')
    .reset_index()
)
players['after5'] = players['first_seen'] + pd.Timedelta(days=5)
players['after15'] = players['first_seen'] + pd.Timedelta(days=15)

# 마지막 로그 시각을 수집 종료 시각으로 간주합니다.
players = players.loc[players['after15'] <= logs['played_at'].max()].copy()
logs_with_windows = logs.merge(players, on='device', how='inner', validate='many_to_one')


### 첫 5일 피처 집계


In [3]:
# 예측 시점 이전 기록만 사용
observation_logs = logs_with_windows.loc[
    logs_with_windows['played_at'] < logs_with_windows['after5']
]
features = observation_logs.groupby('device').agg(
    play_count=('played_at', 'size'),
    mean_score=('score', 'mean'),
    first_play=('played_at', 'min'),
    last_play=('played_at', 'max'),
)
# 초 단위 간격을 일 단위로 환산
features['active_duration_days'] = (
    (features['last_play'] - features['first_play']).dt.total_seconds() / 86400
)
features = features.drop(columns=['first_play', 'last_play']).reset_index()


### 미복귀 라벨·피처 병합


In [4]:
# 5일 경계는 포함, 15일 경계는 제외
outcome_logs = logs_with_windows.loc[
    (logs_with_windows['played_at'] >= logs_with_windows['after5'])
    & (logs_with_windows['played_at'] < logs_with_windows['after15'])
]
# 복귀 기록이 없는 기기를 미복귀(1)로 표시
players['churn'] = (~players['device'].isin(outcome_logs['device'])).astype('int64')
dataset = players.merge(features, on='device', how='left', validate='one_to_one')


### 최종 저장·요약


In [5]:
# 행 인덱스를 제외하고 저장
dataset.to_csv('../data/processed/data.csv', index=False)
display(dataset.head())
print('플레이어 수:', len(dataset))
print('미복귀 비율:', dataset['churn'].mean())


,device,first_seen,after5,after15,churn,play_count,mean_score,active_duration_days
0,0,2015-04-11 11:52:38+00:00,2015-04-16 11:52:38+00:00,2015-04-26 11:52:38+00:00,1,3,58.000000,0.001319
1,000000000000000,2015-01-23 02:04:44+00:00,2015-01-28 02:04:44+00:00,2015-02-07 02:04:44+00:00,1,3,3.333333,0.000741
2,010941525590041,2015-07-07 10:33:00+00:00,2015-07-12 10:33:00+00:00,2015-07-22 10:33:00+00:00,1,1,21.000000,0.000000
3,012345678901237,2015-02-23 03:24:43+00:00,2015-02-28 03:24:43+00:00,2015-03-10 03:24:43+00:00,1,2,32.500000,0.000602
4,012345678912345,2015-02-22 08:51:46+00:00,2015-02-27 08:51:46+00:00,2015-03-09 08:51:46+00:00,1,5,11.200000,0.000544


플레이어 수: 25655
미복귀 비율: 0.9252387448840382
